# AWS Glue Studio Notebook
##### You are now running a AWS Glue Studio notebook; To start using your notebook you need to start an AWS Glue Interactive Session.


#### Optional: Run this cell to see available notebook commands ("magics").


In [ ]:
%help

####  Run this cell to set up and start your interactive session.


In [1]:
%idle_timeout 2880
%glue_version 5.0
%worker_type G.1X
%number_of_workers 5

import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job
  
sc = SparkContext.getOrCreate()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
job = Job(glueContext)

Welcome to the Glue Interactive Sessions Kernel
For more information on available magic commands, please type %help in any new cell.

Please view our Getting Started page to access the most up-to-date information on the Interactive Sessions kernel: https://docs.aws.amazon.com/glue/latest/dg/interactive-sessions.html
Installed kernel version: 1.0.8 
Current idle_timeout is None minutes.
idle_timeout has been set to 2880 minutes.
Setting Glue version to: 5.0
Previous worker type: None
Setting new worker type to: G.1X
Previous number of workers: None
Setting new number of workers to: 5
Trying to create a Glue session for the kernel.
Session Type: glueetl
Worker Type: G.1X
Number of Workers: 5
Idle Timeout: 2880
Session ID: d5286f09-510b-4e62-b6af-a51810502249
Applying the following default arguments:
--glue_kernel_version 1.0.8
--enable-glue-datacatalog true
Waiting for session d5286f09-510b-4e62-b6af-a51810502249 to get into ready status...
Session d5286f09-510b-4e62-b6af-a51810502249 ha

### Unused Columns

In [ ]:
unused_listing_columns = [
    'source',
    'currency'
]

unused_apartment_attributes_columns = [
    'category',
    'body',
    'has_photo',
    'pets_allowed',
    'price_display',
    'square_feet',
    'address',
    'state',
    'latitude',
    'longitude'
]

unused_user_viewing_columns = [
    'viewed_at',  
    'is_wishlisted',  
    'call_to_action'  
]
data_catalog_db = "project-3"
tables_prefix = "raw-"
bucket_name = "bucket"
logger = glueContext.get_logger()

#### Loading Data


In [16]:
# Read from Glue Data Catalog with logging
listing_df = glueContext.create_dynamic_frame.from_catalog(
    database=data_catalog_db,
    table_name=f"{tables_prefix}apartments_csv"
).toDF()

apartment_attribute_df = glueContext.create_dynamic_frame.from_catalog(
    database=data_catalog_db,
    table_name=f"{tables_prefix}apartments_attributes_csv"
).toDF()

bookings_df = glueContext.create_dynamic_frame.from_catalog(
    database=data_catalog_db,
    table_name=f"{tables_prefix}bookings_csv"
).toDF()

user_viewing = glueContext.create_dynamic_frame.from_catalog(
    database=data_catalog_db,
    table_name=f"{tables_prefix}user_viewings_csv"
).toDF()

/usr/lib/spark/python/lib/pyspark.zip/pyspark/sql/dataframe.py:147: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


##### Conversions

In [17]:
from pyspark.sql.functions import to_date, col
from pyspark.sql.types import DateType

# Convert date columns with logging
listing_df = listing_df.withColumn('listing_created_on', col('listing_created_on').cast(DateType()))
listing_df = listing_df.withColumn('last_modified_timestamp', col('last_modified_timestamp').cast(DateType()))

bookings_df = bookings_df.withColumn('booking_date', col('booking_date').cast(DateType()))
bookings_df = bookings_df.withColumn('checkin_date', col('checkin_date').cast(DateType()))
bookings_df = bookings_df.withColumn('checkout_date', col('checkout_date').cast(DateType()))

In [18]:
# Functions 
def drop_unused_cols(df, columns, table_name):
    logger.info(f"Starting drop_unused_cols for {table_name}")
    df = df.drop(*columns)
    logger.info(f"Schema after dropping columns for {table_name}:")
    df.printSchema()
    return df
    
def drop_dupli_null(df, table_name):
    logger.info(f"Starting drop_dupli_null for {table_name}")
    df = df.dropna()
    df = df.dropDuplicates()
    return df

##### Function

##### Process Files

In [19]:
# Process DataFrames with functions and logging
logger.info("Processing listing_df")
listing_df = drop_unused_cols(listing_df, unused_listing_columns, "listing_df")
listing_df = drop_dupli_null(listing_df, "listing_df")

logger.info("Processing apartment_attribute_df")
apartment_attribute_df = drop_unused_cols(apartment_attribute_df, unused_apartment_attributes_columns, "apartment_attribute_df")
apartment_attribute_df = drop_dupli_null(apartment_attribute_df, "apartment_attribute_df")

logger.info("Processing user_viewing")
user_viewing = drop_unused_cols(user_viewing, unused_user_viewing_columns, "user_viewing")
user_viewing = drop_dupli_null(user_viewing, "user_viewing")

logger.info("Processing bookings_df")
bookings_df = drop_dupli_null(bookings_df, "bookings_df")

root
 |-- id: long (nullable = true)
 |-- title: string (nullable = true)
 |-- price: double (nullable = true)
 |-- listing_created_on: date (nullable = true)
 |-- is_active: long (nullable = true)
 |-- last_modified_timestamp: date (nullable = true)

root
 |-- id: long (nullable = true)
 |-- amenities: string (nullable = true)
 |-- bathrooms: long (nullable = true)
 |-- bedrooms: long (nullable = true)
 |-- fee: double (nullable = true)
 |-- price_type: string (nullable = true)
 |-- cityname: string (nullable = true)

root
 |-- user_id: long (nullable = true)
 |-- apartment_id: long (nullable = true)


In [24]:
# Save to S3 as parquet 
listing_df.write.mode("overwrite").parquet(f"s3://{bucket_name}/curated/listings/")

apartment_attribute_df.write.mode("overwrite").parquet(f"s3://{bucket_name}/curated/apartment_attributes/")

bookings_df.write.mode("overwrite").parquet(f"s3://{bucket_name}/curated/bookings/")

user_viewing.write.mode("overwrite").parquet(f"s3://{bucket_name}/curated/user_viewing/")

#### Example: Write the data in the DynamicFrame to a location in Amazon S3 and a table for it in the AWS Glue Data Catalog
